In [ ]:
# ============================================================
# Figure7
# GSE208307 scRNA validation of RLP3/RLP4-associated malignant states
# ============================================================

import os
import tarfile
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from scipy.stats import mannwhitneyu

# ============================================================
# 0. Path config
# ============================================================

PROJECT_DIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC"
FIG7_DIR = os.path.join(PROJECT_DIR, "07_Figure7_GSE208307")
DATA_DIR = FIG7_DIR

RAW_TAR = os.path.join(DATA_DIR, "GSE208307_RAW.tar")
RAW_DIR = os.path.join(DATA_DIR, "GSE208307_RAW")

FIG2_DIR = os.path.join(PROJECT_DIR, "02_Figure2_python")
NMF_TOP_GENE_FILE = os.path.join(
    FIG2_DIR,
    "Fig2_NMF_top_genes_per_Rloop_program.csv"
)

OUT_DIR = os.path.join(FIG7_DIR, "figures")
TAB_DIR = os.path.join(FIG7_DIR, "tables")
OBJ_DIR = os.path.join(FIG7_DIR, "objects")

for d in [RAW_DIR, OUT_DIR, TAB_DIR, OBJ_DIR]:
    os.makedirs(d, exist_ok=True)

# 重要：如果作者确认 296/297 的 response 相反，只改这里
RESPONSE_MAP = {
    "scRNAseq-296": "Responder",
    "scRNAseq-297": "Resistant"
}

sc.settings.figdir = OUT_DIR
sc.set_figure_params(figsize=(5, 4), dpi=120)

def save_fig(fig, name, width=5, height=4):
    fig.set_size_inches(width, height)
    fig.savefig(os.path.join(OUT_DIR, f"{name}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(OUT_DIR, f"{name}.png"), dpi=600, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", name)

In [ ]:
# ============================================================
# 1. Extract GEO raw tar
# ============================================================

def extract_tar_recursive(tar_path, out_dir):
    if not os.path.exists(tar_path):
        raise FileNotFoundError(tar_path)

    with tarfile.open(tar_path, "r:*") as tar:
        tar.extractall(out_dir)

    # 递归解压内部 tar / tar.gz
    inner_tars = glob.glob(os.path.join(out_dir, "**", "*.tar*"), recursive=True)
    for t in inner_tars:
        try:
            inner_dir = t.replace(".tar.gz", "").replace(".tar", "")
            os.makedirs(inner_dir, exist_ok=True)
            with tarfile.open(t, "r:*") as tar:
                tar.extractall(inner_dir)
        except Exception as e:
            print("Skip:", t, e)

extract_tar_recursive(RAW_TAR, RAW_DIR)

In [ ]:
# ============================================================
# 2. Find 10x folders
# ============================================================

def find_10x_dirs(root):
    dirs = []
    for path, subdirs, files in os.walk(root):
        files_lower = [f.lower() for f in files]
        has_matrix = any(f.startswith("matrix.mtx") for f in files_lower)
        has_barcodes = any(f.startswith("barcodes.tsv") for f in files_lower)
        has_features = any(f.startswith("features.tsv") or f.startswith("genes.tsv") for f in files_lower)

        if has_matrix and has_barcodes and has_features:
            dirs.append(path)
    return sorted(set(dirs))

tenx_dirs = find_10x_dirs(RAW_DIR)
print("10x dirs found:")
for d in tenx_dirs:
    print(d)

In [ ]:
# ============================================================
# 3. Read and merge samples
# ============================================================

adatas = []

for d in tenx_dirs:
    sample_name = None
    for key in ["scRNAseq-296", "scRNAseq-297", "GSM6340558", "GSM6340559"]:
        if key in d:
            sample_name = key
            break

    if sample_name is None:
        sample_name = os.path.basename(d)

    if sample_name == "GSM6340558":
        sample_name = "scRNAseq-296"
    if sample_name == "GSM6340559":
        sample_name = "scRNAseq-297"

    ad = sc.read_10x_mtx(d, var_names="gene_symbols", cache=False)
    ad.var_names_make_unique()

    ad.obs["sample"] = sample_name
    ad.obs["response"] = RESPONSE_MAP.get(sample_name, "Unknown")

    adatas.append(ad)

adata = adatas[0].concatenate(
    adatas[1:],
    batch_key="batch",
    batch_categories=[a.obs["sample"].iloc[0] for a in adatas]
)

adata.obs["sample"] = adata.obs["batch"].astype(str)
adata.obs["response"] = adata.obs["sample"].map(RESPONSE_MAP)

print(adata)
print(adata.obs[["sample", "response"]].drop_duplicates())

In [ ]:
# ============================================================
# 4. QC and preprocessing
# ============================================================

adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

# 可根据数据情况调整
adata = adata[
    (adata.obs["n_genes_by_counts"] > 200) &
    (adata.obs["n_genes_by_counts"] < 7000) &
    (adata.obs["pct_counts_mt"] < 20),
    :
].copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata.raw = adata.copy()

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    batch_key="sample"
)

adata = adata[:, adata.var["highly_variable"]].copy()

sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")
sc.pp.neighbors(adata, n_neighbors=20, n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.6)

adata.write(os.path.join(OBJ_DIR, "GSE208307_processed.h5ad"))

In [ ]:
# ============================================================
# 5. Cell type annotation by marker scores
# ============================================================

marker_sets = {
    "Malignant": ["EPCAM", "KRT8", "KRT18", "KRT19", "ALB", "AFP"],
    "T_NK": ["CD3D", "CD3E", "TRAC", "CD8A", "CD8B", "NKG7", "GNLY"],
    "Treg": ["FOXP3", "IL2RA", "CTLA4", "TIGIT"],
    "Myeloid": ["LYZ", "LST1", "TYROBP", "FCER1G", "C1QA", "C1QB", "C1QC"],
    "B_Plasma": ["MS4A1", "CD79A", "CD79B", "MZB1", "JCHAIN"],
    "Endothelial": ["PECAM1", "VWF", "KDR", "EMCN"],
    "Fibroblast": ["COL1A1", "COL1A2", "DCN", "LUM", "ACTA2"]
}

for ct, genes in marker_sets.items():
    genes_use = [g for g in genes if g in adata.raw.var_names]
    if len(genes_use) >= 2:
        sc.tl.score_genes(
            adata,
            gene_list=genes_use,
            score_name=f"{ct}_score",
            use_raw=True
        )

score_cols = [f"{ct}_score" for ct in marker_sets if f"{ct}_score" in adata.obs.columns]

score_mat = adata.obs[score_cols].copy()
score_mat.columns = [c.replace("_score", "") for c in score_mat.columns]

adata.obs["celltype_auto"] = score_mat.idxmax(axis=1)

# Treg 优先覆盖
if "Treg_score" in adata.obs.columns:
    adata.obs.loc[
        adata.obs["Treg_score"] > np.percentile(adata.obs["Treg_score"], 90),
        "celltype_auto"
    ] = "Treg"

print(adata.obs["celltype_auto"].value_counts())

adata.write(os.path.join(OBJ_DIR, "GSE208307_celltype_annotated.h5ad"))

In [ ]:
# ============================================================
# 6. Load RLP3 / RLP4 genes and score cells
# ============================================================

nmf_genes = pd.read_csv(NMF_TOP_GENE_FILE)

rlp3_genes = (
    nmf_genes.loc[nmf_genes["program"] == "RLP3", "gene"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

rlp4_genes = (
    nmf_genes.loc[nmf_genes["program"] == "RLP4", "gene"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("RLP3 genes:", len(rlp3_genes))
print("RLP4 genes:", len(rlp4_genes))

for name, genes in {
    "RLP3_score": rlp3_genes,
    "RLP4_score": rlp4_genes
}.items():

    genes_use = [g for g in genes if g in adata.raw.var_names]

    print(name, "matched:", len(genes_use))

    if len(genes_use) < 2:
        raise ValueError(f"Too few matched genes for {name}")

    sc.tl.score_genes(
        adata,
        gene_list=genes_use,
        score_name=name,
        use_raw=True
    )

score_z = StandardScaler().fit_transform(
    adata.obs[["RLP3_score", "RLP4_score"]]
)

adata.obs["RLP3_score_z"] = score_z[:, 0]
adata.obs["RLP4_score_z"] = score_z[:, 1]
adata.obs["RLP3_RLP4_combined_score"] = score_z.mean(axis=1)

# 只在 malignant cells 中定义 high/low
mal_mask = adata.obs["celltype_auto"] == "Malignant"

cutoff = adata.obs.loc[
    mal_mask,
    "RLP3_RLP4_combined_score"
].median()

adata.obs["RLP3_RLP4_group"] = "Non-malignant"

adata.obs.loc[
    mal_mask & (adata.obs["RLP3_RLP4_combined_score"] >= cutoff),
    "RLP3_RLP4_group"
] = "RLP3/RLP4-high malignant"

adata.obs.loc[
    mal_mask & (adata.obs["RLP3_RLP4_combined_score"] < cutoff),
    "RLP3_RLP4_group"
] = "RLP3/RLP4-low malignant"

# 删除错误生成的列
if "RLP3/RLP4-high malignant" in adata.obs.columns:
    adata.obs = adata.obs.drop(columns=["RLP3/RLP4-high malignant"])

# 确保 obs 中 object 类型都能保存
for col in adata.obs.columns:
    if adata.obs[col].dtype == "object":
        adata.obs[col] = adata.obs[col].astype(str)

print(adata.obs["RLP3_RLP4_group"].value_counts())

adata.write(os.path.join(OBJ_DIR, "GSE208307_RLP_scored.h5ad"))

In [ ]:
# ============================================================
# Make PDF/SVG text editable in Adobe Illustrator
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt
import os

mpl.rcParams["pdf.fonttype"] = 42      # TrueType, editable text in AI
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"  # keep text as text in SVG
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

In [ ]:
# ============================================================
# Figure7A
# UMAP overview
# Editable text version for Adobe Illustrator
# ============================================================

# ----------------------------
# sample / response / celltype
# ----------------------------

sc.pl.umap(
    adata,
    color=["sample", "response", "celltype_auto"],
    wspace=0.35,
    show=False,
    frameon=False,
    size=8
)

fig = plt.gcf()

fig.savefig(
    os.path.join(OUT_DIR, "Figure7A_UMAP_sample_response_celltype_editable.pdf"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7A_UMAP_sample_response_celltype_editable.svg"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7A_UMAP_sample_response_celltype.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)


# ----------------------------
# RLP3/RLP4 score / group
# ----------------------------

sc.pl.umap(
    adata,
    color=["RLP3_RLP4_combined_score", "RLP3_RLP4_group"],
    cmap="RdBu_r",
    wspace=0.35,
    show=False,
    frameon=False,
    size=8
)

fig = plt.gcf()

fig.savefig(
    os.path.join(OUT_DIR, "Figure7A_UMAP_RLP3_RLP4_state_editable.pdf"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7A_UMAP_RLP3_RLP4_state_editable.svg"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7A_UMAP_RLP3_RLP4_state.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)

In [ ]:
# ============================================================
# Figure7B improved
# 100% stacked barplot + statistics table
# AI editable version
# ============================================================

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import os

# ------------------------------------------------------------
# Make text editable in Adobe Illustrator
# ------------------------------------------------------------
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

# ------------------------------------------------------------
# 1. Only malignant cells
# ------------------------------------------------------------
mal = adata[adata.obs["celltype_auto"] == "Malignant"].copy()

count_df = (
    mal.obs
    .groupby(["sample", "response", "RLP3_RLP4_group"])
    .size()
    .reset_index(name="n")
)

total_df = (
    mal.obs
    .groupby(["sample", "response"])
    .size()
    .reset_index(name="total_malignant_cells")
)

prop_df = count_df.merge(total_df, on=["sample", "response"])
prop_df["proportion"] = prop_df["n"] / prop_df["total_malignant_cells"] * 100

prop_df = prop_df[
    prop_df["RLP3_RLP4_group"].isin([
        "RLP3/RLP4-high malignant",
        "RLP3/RLP4-low malignant"
    ])
].copy()

# ------------------------------------------------------------
# 2. Build statistics table
# ------------------------------------------------------------
stat_rows = []

for sample in sorted(prop_df["sample"].unique()):
    tmp = prop_df[prop_df["sample"] == sample]

    response = tmp["response"].iloc[0]
    total = tmp["total_malignant_cells"].iloc[0]

    high_n = tmp.loc[
        tmp["RLP3_RLP4_group"] == "RLP3/RLP4-high malignant",
        "n"
    ].sum()

    low_n = tmp.loc[
        tmp["RLP3_RLP4_group"] == "RLP3/RLP4-low malignant",
        "n"
    ].sum()

    high_pct = high_n / total * 100
    low_pct = low_n / total * 100

    stat_rows.append({
        "Sample": sample,
        "Response": response,
        "Malignant cells": int(total),
        "Combined-high n": int(high_n),
        "Combined-high %": high_pct,
        "Combined-low n": int(low_n),
        "Combined-low %": low_pct
    })

stat_table = pd.DataFrame(stat_rows)

# 排序：Resistant 左，Responder 右
response_order = {"Resistant": 0, "Responder": 1}
stat_table["response_order"] = stat_table["Response"].map(response_order)
stat_table = stat_table.sort_values(["response_order", "Sample"]).reset_index(drop=True)

stat_table.to_csv(
    os.path.join(TAB_DIR, "Figure7B_RLP3_RLP4_malignant_state_statistics.csv"),
    index=False
)

display(stat_table)

In [ ]:
# ============================================================
# 3. Plot AI editable stacked barplot
# ============================================================

colors = {
    "Combined-low": "#4E79A7",
    "Combined-high": "#E15759"
}

fig, ax = plt.subplots(figsize=(4.8, 4.4))

x = np.arange(stat_table.shape[0])

low_vals = stat_table["Combined-low %"].values
high_vals = stat_table["Combined-high %"].values

# low part
ax.bar(
    x,
    low_vals,
    color=colors["Combined-low"],
    edgecolor="black",
    linewidth=0.6,
    label="Combined-low"
)

# high part
ax.bar(
    x,
    high_vals,
    bottom=low_vals,
    color=colors["Combined-high"],
    edgecolor="black",
    linewidth=0.6,
    label="Combined-high"
)

# percentage labels
for i in range(len(x)):
    if low_vals[i] > 8:
        ax.text(
            x[i],
            low_vals[i] / 2,
            f"{low_vals[i]:.1f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white"
        )

    if high_vals[i] > 8:
        ax.text(
            x[i],
            low_vals[i] + high_vals[i] / 2,
            f"{high_vals[i]:.1f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white"
        )

# x labels
xlabels = [
    f"{row['Sample']}\n({row['Response']})"
    for _, row in stat_table.iterrows()
]

ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=11)

ax.set_ylim(0, 100)
ax.set_ylabel("Proportion of malignant cells (%)", fontsize=12)
ax.set_xlabel("")

ax.set_title(
    "Composition of combined RLP3-RLP4 malignant states",
    fontsize=13,
    fontweight="bold"
)

ax.legend(
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=11
)

ax.tick_params(axis="y", labelsize=11)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(axis="y", linestyle="-", alpha=0.25)

# ------------------------------------------------------------
# Save editable vector files
# ------------------------------------------------------------

fig.savefig(
    os.path.join(OUT_DIR, "Figure7B_RLP3_RLP4_malignant_state_stacked_bar_editable.pdf"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7B_RLP3_RLP4_malignant_state_stacked_bar_editable.svg"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7B_RLP3_RLP4_malignant_state_stacked_bar.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)

In [ ]:
# ============================================================
# Plot Figure7B
# AI editable version
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os

# 让 PDF/SVG 中的文字保持为可编辑文字
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

colors = {
    "RLP3/RLP4-low malignant": "#4E79A7",
    "RLP3/RLP4-high malignant": "#E15759"
}

fig, ax = plt.subplots(figsize=(5.2, 4.5))

x = np.arange(plot_df.shape[0])

low_vals = plot_df["RLP3/RLP4-low malignant"].values
high_vals = plot_df["RLP3/RLP4-high malignant"].values

ax.bar(
    x,
    low_vals,
    color=colors["RLP3/RLP4-low malignant"],
    edgecolor="black",
    linewidth=0.6,
    label="Combined-low"
)

ax.bar(
    x,
    high_vals,
    bottom=low_vals,
    color=colors["RLP3/RLP4-high malignant"],
    edgecolor="black",
    linewidth=0.6,
    label="Combined-high"
)

for i in range(len(x)):
    if low_vals[i] > 8:
        ax.text(
            x[i],
            low_vals[i] / 2,
            f"{low_vals[i]:.1f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white"
        )

    if high_vals[i] > 8:
        ax.text(
            x[i],
            low_vals[i] + high_vals[i] / 2,
            f"{high_vals[i]:.1f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white"
        )

xlabels = [
    f"{row['sample']}\n({row['response']})"
    for _, row in plot_df.iterrows()
]

ax.set_xticks(x)
ax.set_xticklabels(xlabels, rotation=0, fontsize=11)

ax.set_ylim(0, 100)
ax.set_ylabel("Proportion of malignant cells (%)", fontsize=12)
ax.set_xlabel("")

ax.set_title(
    "Composition of combined RLP3-RLP4 malignant states",
    fontsize=13,
    fontweight="bold"
)

ax.legend(
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=11
)

ax.tick_params(axis="y", labelsize=11)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(axis="y", linestyle="-", alpha=0.25)

# 保存 AI 可编辑版本
fig.savefig(
    os.path.join(OUT_DIR, "Figure7B_RLP3_RLP4_malignant_state_stacked_bar_editable.pdf"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7B_RLP3_RLP4_malignant_state_stacked_bar_editable.svg"),
    bbox_inches="tight"
)

# 同时保存高清 PNG 预览
fig.savefig(
    os.path.join(OUT_DIR, "Figure7B_RLP3_RLP4_malignant_state_stacked_bar.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)

In [ ]:
# ============================================================
# 7. Functional signature scores in malignant cells
# ============================================================

functional_signatures = {
    "Hypoxia": ["HIF1A", "VEGFA", "CA9", "SLC2A1", "LDHA", "ENO1", "PGK1", "BNIP3"],
    "Glycolysis": ["HK2", "PFKP", "ALDOA", "GAPDH", "PGK1", "ENO1", "PKM", "LDHA"],
    "EMT": ["VIM", "FN1", "SNAI1", "SNAI2", "ZEB1", "ZEB2", "TWIST1", "COL1A1"],
    "MHC_I": ["HLA-A", "HLA-B", "HLA-C", "B2M", "TAP1", "TAP2"],
    "IFNG_Response": ["IFNG", "STAT1", "IRF1", "CXCL9", "CXCL10", "CXCL11", "IDO1"],
    "Antigen_presentation": ["HLA-A", "HLA-B", "HLA-C", "B2M", "TAP1", "TAP2", "NLRC5"],
    "Cell_cycle": ["MKI67", "TOP2A", "PCNA", "MCM2", "MCM5", "CDK1", "CCNB1"],
    "Stress_UPR": ["HSPA5", "XBP1", "DDIT3", "ATF4", "HSP90B1", "HERPUD1"]
}

for sig, genes in functional_signatures.items():
    genes_use = [g for g in genes if g in adata.raw.var_names]
    if len(genes_use) >= 2:
        sc.tl.score_genes(
            adata,
            gene_list=genes_use,
            score_name=sig,
            use_raw=True
        )

mal = adata[adata.obs["celltype_auto"] == "Malignant"].copy()

sig_cols = [s for s in functional_signatures if s in mal.obs.columns]

mal_score_df = mal.obs[
    ["sample", "response", "RLP3_RLP4_group", "RLP3_RLP4_combined_score"] + sig_cols
].copy()

mal_score_df.to_csv(
    os.path.join(TAB_DIR, "Figure7C_malignant_functional_scores.csv")
)

In [ ]:
# ============================================================
# Figure7C improved
# Functional states in combined RLP3-RLP4 high vs low malignant cells
# violin + boxplot + statistics
# AI editable version
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from scipy.stats import mannwhitneyu

# ------------------------------------------------------------
# Editable text in Adobe Illustrator
# ------------------------------------------------------------
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

# ------------------------------------------------------------
# Prepare long table
# ------------------------------------------------------------
plot_long = mal_score_df.melt(
    id_vars=["sample", "response", "RLP3_RLP4_group"],
    value_vars=sig_cols,
    var_name="Signature",
    value_name="Score"
)

plot_long = plot_long[
    plot_long["RLP3_RLP4_group"].isin([
        "RLP3/RLP4-low malignant",
        "RLP3/RLP4-high malignant"
    ])
].copy()

plot_long["RLP3_RLP4_group_short"] = plot_long["RLP3_RLP4_group"].replace({
    "RLP3/RLP4-low malignant": "Combined-low",
    "RLP3/RLP4-high malignant": "Combined-high"
})

plot_long["RLP3_RLP4_group_short"] = pd.Categorical(
    plot_long["RLP3_RLP4_group_short"],
    categories=["Combined-low", "Combined-high"],
    ordered=True
)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------
stat_rows = []

for sig in sig_cols:
    tmp = plot_long[plot_long["Signature"] == sig]

    low = tmp.loc[tmp["RLP3_RLP4_group_short"] == "Combined-low", "Score"]
    high = tmp.loc[tmp["RLP3_RLP4_group_short"] == "Combined-high", "Score"]

    stat, pval = mannwhitneyu(low, high, alternative="two-sided")

    low_median = low.median()
    high_median = high.median()
    delta_median = high_median - low_median

    stat_rows.append({
        "Signature": sig,
        "Low_n": len(low),
        "High_n": len(high),
        "Low_median": low_median,
        "High_median": high_median,
        "Delta_median_high_minus_low": delta_median,
        "P_value": pval
    })

stat_df = pd.DataFrame(stat_rows)

stat_df["Significance"] = stat_df["P_value"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
)

stat_df.to_csv(
    os.path.join(TAB_DIR, "Figure7C_functional_state_statistics.csv"),
    index=False
)

display(stat_df)

In [ ]:
# ============================================================
# Figure7C refined
# Functional states in combined RLP3-RLP4 high vs low malignant cells
# violin + narrow boxplot + statistics
# AI editable version
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from scipy.stats import mannwhitneyu

# ------------------------------------------------------------
# Editable text in Adobe Illustrator
# ------------------------------------------------------------
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

palette = {
    "Combined-low": "#4E79A7",
    "Combined-high": "#E15759"
}

# ------------------------------------------------------------
# Prepare long table
# ------------------------------------------------------------
plot_long = mal_score_df.melt(
    id_vars=["sample", "response", "RLP3_RLP4_group"],
    value_vars=sig_cols,
    var_name="Signature",
    value_name="Score"
)

plot_long = plot_long[
    plot_long["RLP3_RLP4_group"].isin([
        "RLP3/RLP4-low malignant",
        "RLP3/RLP4-high malignant"
    ])
].copy()

plot_long["RLP3_RLP4_group_short"] = plot_long["RLP3_RLP4_group"].replace({
    "RLP3/RLP4-low malignant": "Combined-low",
    "RLP3/RLP4-high malignant": "Combined-high"
})

plot_long["RLP3_RLP4_group_short"] = pd.Categorical(
    plot_long["RLP3_RLP4_group_short"],
    categories=["Combined-low", "Combined-high"],
    ordered=True
)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------
stat_rows = []

for sig in sig_cols:
    tmp = plot_long[plot_long["Signature"] == sig]

    low = tmp.loc[tmp["RLP3_RLP4_group_short"] == "Combined-low", "Score"]
    high = tmp.loc[tmp["RLP3_RLP4_group_short"] == "Combined-high", "Score"]

    stat, pval = mannwhitneyu(low, high, alternative="two-sided")

    low_median = low.median()
    high_median = high.median()

    stat_rows.append({
        "Signature": sig,
        "Low_n": len(low),
        "High_n": len(high),
        "Low_median": low_median,
        "High_median": high_median,
        "Delta_median_high_minus_low": high_median - low_median,
        "P_value": pval
    })

stat_df = pd.DataFrame(stat_rows)

stat_df["Significance"] = stat_df["P_value"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
)

stat_df.to_csv(
    os.path.join(TAB_DIR, "Figure7C_functional_state_statistics.csv"),
    index=False
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.5, 4.8))

# 1. violin：更宽、更浅
sns.violinplot(
    data=plot_long,
    x="Signature",
    y="Score",
    hue="RLP3_RLP4_group_short",
    hue_order=["Combined-low", "Combined-high"],
    split=True,
    inner=None,
    palette=palette,
    linewidth=0.6,
    cut=0,
    scale="width",
    saturation=0.75,
    ax=ax
)

# 降低 violin 透明度
for coll in ax.collections:
    try:
        coll.set_alpha(0.65)
    except Exception:
        pass

# 2. boxplot：更窄，不压住 violin
sns.boxplot(
    data=plot_long,
    x="Signature",
    y="Score",
    hue="RLP3_RLP4_group_short",
    hue_order=["Combined-low", "Combined-high"],
    width=0.12,
    dodge=True,
    showcaps=True,
    showfliers=False,
    boxprops={
        "facecolor": "white",
        "edgecolor": "black",
        "linewidth": 0.7,
        "zorder": 4,
        "alpha": 0.85
    },
    whiskerprops={"linewidth": 0.7},
    capprops={"linewidth": 0.7},
    medianprops={"color": "black", "linewidth": 1.1},
    ax=ax
)

# 3. 去掉重复 legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=11,
    title=""
)

# 4. 统计标注：更紧凑
global_ymin = plot_long["Score"].quantile(0.001)
global_ymax = plot_long["Score"].quantile(0.995)
yrange = global_ymax - global_ymin

ymax_by_sig = plot_long.groupby("Signature")["Score"].quantile(0.995).to_dict()

for i, sig in enumerate(sig_cols):
    sig_stat = stat_df[stat_df["Signature"] == sig].iloc[0]

    sig_label = sig_stat["Significance"]
    delta = sig_stat["Delta_median_high_minus_low"]

    y = ymax_by_sig[sig] + yrange * 0.05
    h = yrange * 0.018

    x1 = i - 0.18
    x2 = i + 0.18

    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        color="black",
        linewidth=0.75
    )

    ax.text(
        i,
        y + h + yrange * 0.012,
        sig_label,
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

    ax.text(
        i,
        y + h + yrange * 0.055,
        f"Δ={delta:.2f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

ax.set_title(
    "Functional states of combined RLP3-RLP4 malignant cells",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("")
ax.set_ylabel("Signature score", fontsize=12)

plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
    fontsize=10
)

ax.tick_params(axis="y", labelsize=10)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(axis="y", linestyle="-", alpha=0.18)

ax.set_ylim(
    global_ymin - yrange * 0.08,
    max(ymax_by_sig.values()) + yrange * 0.22
)

# ------------------------------------------------------------
# Save editable vector files
# ------------------------------------------------------------
fig.savefig(
    os.path.join(OUT_DIR, "Figure7C_RLP_high_low_malignant_functional_states_refined_editable.pdf"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7C_RLP_high_low_malignant_functional_states_refined_editable.svg"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7C_RLP_high_low_malignant_functional_states_refined.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)

In [ ]:
# ============================================================
# Figure7D refined
# Representative immune-metabolic genes in combined RLP3-RLP4 malignant cells
# AI editable text version
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt
import os

# ------------------------------------------------------------
# Make text editable in Adobe Illustrator
# ------------------------------------------------------------
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

# ------------------------------------------------------------
# Representative genes
# ------------------------------------------------------------
dot_genes = [
    # Hypoxia / glycolysis
    "HIF1A", "VEGFA", "LDHA", "ENO1", "SLC2A1",

    # Antigen presentation / MHC-I
    "HLA-A", "HLA-B", "B2M", "TAP1",

    # Immune suppression / checkpoint-related
    "MIF", "LGALS9", "CD274",

    # Stress / UPR
    "DDIT3", "XBP1",

    # EMT / aggressive malignant state
    "VIM", "SPP1"
]

dot_genes = [g for g in dot_genes if g in adata.raw.var_names]

print("Genes used in Figure7D:")
print(dot_genes)

# ------------------------------------------------------------
# Simplify group names
# ------------------------------------------------------------
mal.obs["RLP3_RLP4_group_short"] = mal.obs["RLP3_RLP4_group"].replace({
    "RLP3/RLP4-low malignant": "Combined-low",
    "RLP3/RLP4-high malignant": "Combined-high"
})

# Optional: ensure order
mal.obs["RLP3_RLP4_group_short"] = mal.obs["RLP3_RLP4_group_short"].astype("category")
mal.obs["RLP3_RLP4_group_short"] = mal.obs["RLP3_RLP4_group_short"].cat.reorder_categories(
    ["Combined-low", "Combined-high"],
    ordered=True
)

# ------------------------------------------------------------
# Dotplot
# ------------------------------------------------------------
sc.pl.dotplot(
    mal,
    var_names=dot_genes,
    groupby="RLP3_RLP4_group_short",
    use_raw=True,
    standard_scale="var",
    cmap="RdBu_r",
    dot_max=0.75,
    dot_min=0.03,
    show=False
)

fig = plt.gcf()

fig.suptitle(
    "Representative immune-metabolic genes in combined RLP3-RLP4 malignant cells",
    fontsize=13,
    fontweight="bold",
    y=1.02
)

# ------------------------------------------------------------
# Save editable vector files
# ------------------------------------------------------------
fig.savefig(
    os.path.join(OUT_DIR, "Figure7D_immune_metabolic_gene_dotplot_editable.pdf"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7D_immune_metabolic_gene_dotplot_editable.svg"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7D_immune_metabolic_gene_dotplot.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)

In [ ]:
# ============================================================
# Figure7E refined
# Differential ligand-receptor communication proxy
# Combined-high versus Combined-low
# AI editable version
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# Make text editable in Adobe Illustrator
# ------------------------------------------------------------
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

# ------------------------------------------------------------
# Convert sender names
# ------------------------------------------------------------
comm_df["Sender_short"] = comm_df["Sender"].replace({
    "RLP3/RLP4-low malignant": "Combined-low",
    "RLP3/RLP4-high malignant": "Combined-high"
})

# ------------------------------------------------------------
# Build differential communication table
# ------------------------------------------------------------
wide = comm_df.pivot_table(
    index=["Pair", "Target"],
    columns="Sender_short",
    values="Communication_score",
    fill_value=0
).reset_index()

for col in ["Combined-low", "Combined-high"]:
    if col not in wide.columns:
        wide[col] = 0

wide["Delta_high_minus_low"] = wide["Combined-high"] - wide["Combined-low"]

wide["Log2_ratio_high_vs_low"] = np.log2(
    (wide["Combined-high"] + 1e-6) / (wide["Combined-low"] + 1e-6)
)

wide["Abs_delta"] = wide["Delta_high_minus_low"].abs()

wide.to_csv(
    os.path.join(TAB_DIR, "Figure7E_differential_communication_high_vs_low.csv"),
    index=False
)

plot_df = wide.copy()

# ------------------------------------------------------------
# Order ligand-receptor pairs by maximum absolute delta
# ------------------------------------------------------------
pair_order = (
    plot_df
    .groupby("Pair")["Abs_delta"]
    .max()
    .sort_values(ascending=True)
    .index
    .tolist()
)

target_order = ["Myeloid", "T_NK", "Treg"]

plot_df["Pair"] = pd.Categorical(
    plot_df["Pair"],
    categories=pair_order,
    ordered=True
)

plot_df["Target"] = pd.Categorical(
    plot_df["Target"],
    categories=target_order,
    ordered=True
)

plot_df = plot_df.sort_values(["Pair", "Target"])

# ------------------------------------------------------------
# Bubble size scaling
# ------------------------------------------------------------
max_abs = plot_df["Abs_delta"].max()

if pd.isna(max_abs) or max_abs == 0:
    max_abs = 1.0
    plot_df["Bubble_size"] = 100
else:
    plot_df["Bubble_size"] = 90 + plot_df["Abs_delta"] / max_abs * 560

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.8, 4.8))

scat = ax.scatter(
    x=plot_df["Target"],
    y=plot_df["Pair"],
    s=plot_df["Bubble_size"],
    c=plot_df["Log2_ratio_high_vs_low"],
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    edgecolors="black",
    linewidths=0.45,
    alpha=0.9
)

# ------------------------------------------------------------
# Colorbar
# ------------------------------------------------------------
cbar = plt.colorbar(
    scat,
    ax=ax,
    fraction=0.045,
    pad=0.04
)

cbar.set_label(
    "log2(Combined-high / Combined-low)",
    fontsize=10
)

cbar.ax.tick_params(labelsize=10)

# ------------------------------------------------------------
# Title and labels
# ------------------------------------------------------------
ax.set_title(
    "Differential malignant-to-immune communication",
    fontsize=13,
    fontweight="bold",
    pad=10
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)

# ------------------------------------------------------------
# Clean style
# ------------------------------------------------------------
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(axis="x", linestyle="-", alpha=0.15)
ax.grid(axis="y", linestyle="-", alpha=0.15)

# ------------------------------------------------------------
# Add size legend manually with larger spacing
# ------------------------------------------------------------
size_levels = [0.25, 0.50, 0.75]
legend_handles = []

for frac in size_levels:
    legend_handles.append(
        ax.scatter(
            [],
            [],
            s=90 + frac * 560,
            color="grey",
            edgecolors="black",
            linewidths=0.4,
            alpha=0.8,
            label=f"{frac * max_abs:.2f}"
        )
    )

leg = ax.legend(
    handles=legend_handles,
    title="|Δ score|",
    frameon=False,
    bbox_to_anchor=(1.38, 0.78),
    loc="upper left",
    fontsize=9,
    title_fontsize=10,
    labelspacing=1.8,
    borderpad=0.8,
    handletextpad=1.2,
    scatterpoints=1
)

# Compatible legend handle adjustment for different matplotlib versions
legend_handle_list = []

if hasattr(leg, "legendHandles"):
    legend_handle_list = leg.legendHandles
elif hasattr(leg, "legend_handles"):
    legend_handle_list = leg.legend_handles

for lh in legend_handle_list:
    try:
        lh.set_alpha(0.8)
        lh.set_edgecolor("black")
    except Exception:
        pass

# ------------------------------------------------------------
# Save editable vector files
# ------------------------------------------------------------
fig.savefig(
    os.path.join(OUT_DIR, "Figure7E_differential_communication_high_vs_low_editable.pdf"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7E_differential_communication_high_vs_low_editable.svg"),
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(OUT_DIR, "Figure7E_differential_communication_high_vs_low.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.close(fig)